In [ ]:
%%bash

pip install --upgrade 'scikit-learn<1.6.0'

In [ ]:
#Loading the relevant libraries and functions
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import sklearn
from sklearn import model_selection
from sklearn import preprocessing, pipeline, compose, impute
from sklearn import linear_model, tree
from sklearn import metrics

# Input data files are available in the read-only "../input/" directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# 1. Frame the Problem (15 pts)

Answer the following questions.

1. Define the objective: what are you trying to predict?
2. How should you frame this problem as a machine learning problem?
3. How should performance be measured? 


In [ ]:
#1. The objective is to predict the sale price of houses as a function of the given features.
#2. We are trying to train a model that outputs real numerical values from some features. This is a regression problem, where we are trying
#   to minimize the difference between the model predictions and the target values.
#3. The performance is measured with the mean squred error between the logarithm of the predicted and actual house prices.

# 2. Get the Data (15 pts)

Minimum expectations:

1. Load the data.
2. Check the size and type of data.

In [ ]:
#1. Loading the data.
housing_df = pd.read_csv(
    "/kaggle/input/house-prices-advanced-regression-techniques/train.csv",
    index_col="Id",
)

In [ ]:
#2. Checking the data.
housing_df.head()

In [ ]:
#Checking the data types and dimentions of the dataset.
housing_df.info()

In [ ]:
#Data:
#1460 instances.
#78 features.
#Types: Integer, real numbers, strings.

# 3. Explore the Data (15 pts)

General expectations

1. Discuss the types of features (categorical, ordinal, int/float, bounded/unbounded, text, etc.)
2. What features have missing values?
3. Are there any outliers? If so, how can you tell?
4. What features appear useful for the task? Are there any that appear useless? Why?
5. Identify and characterize the target distribution.
6. Visualize the data.
7. Study the correlations between attributes.


In [ ]:
%%bash
cat /kaggle/input/house-prices-advanced-regression-techniques/data_description.txt
#Data description

In [ ]:
#1. Description of the data.

##### Categorical, String. #####
# MSZoning: Identifies the general zoning classification of the sale.
# Street: Type of road access to property.
# Alley: Type of alley access to property.
# LotShape: General shape of property.
# LandContour: Flatness of the property.
# Utilities: Type of utilities available
# LotConfig: Lot configuration.
# LandSlope: Slope of property.
# Neighborhood: Physical locations within Ames city limits.
# Condition1: Proximity to various conditions.
# Condition2: Proximity to various conditions (if more than one is present).
# BldgType: Type of dwelling,
# HouseStyle: Style of dwelling
# RoofStyle: Type of roof.
# RoofMatl: Roof material.
# Exterior1st: Exterior covering on house.
# Exterior2nd: Exterior covering on house (if more than one material).
# MasVnrType: Masonry veneer type.
# MasVnrArea: Masonry veneer area in square feet.
# Foundation: Type of foundation
# Heating: Type of heating.
# GarageType: Garage location.
# PavedDrive: Paved driveway
# MiscFeature: Miscellaneous feature not covered in other categories.
# SaleType: Type of sale
# SaleCondition: Condition of sale

##### Categorical, Ordinal, String. #####
# BsmtQual: Evaluates the height of the basement
# BsmtCond: Evaluates the general condition of the basement.
# BsmtExposure: Refers to walkout or garden level walls.
# BsmtFinType1: Rating of basement finished area.
# BsmtFinType2: Rating of basement finished area (if multiple types).
# HeatingQC: Heating quality and condition.
# CentralAir: Central air conditioning.
# Electrical: Electrical system.
# KitchenQual: Kitchen quality.
# Functional: Home functionality (Assume typical unless deductions are warranted).
# FireplaceQu: Fireplace quality.
# GarageFinish: Interior finish of the garage.
# GarageQual: Garage quality
# GarageCond: Garage condition
# PoolQC: Pool quality
# Fence: Fence quality

##### Categorical, Ordinal, Integer, Bounded. #####
# OverallQual: Rates the overall material and finish of the house.
# OverallCond: Rates the overall condition of the house.
# MSSubClass: Identifies the type of dwelling involved in the sale.
# ExterQual: Evaluates the quality of the material on the exterior 
# ExterCond: Evaluates the present condition of the material on the exterior

##### Numerical, Float, Unbounded. #####
# LotFrontage: Linear feet of street connected to property.
# LotArea: Lot size in square feet.
# BsmtFinSF1: Type 1 finished square feet.
# BsmtFinSF2: Type 2 finished square feet.
# BsmtUnfSF: Unfinished square feet of basement area
# TotalBsmtSF: Total square feet of basement area
# 1stFlrSF: First Floor square feet
# 2ndFlrSF: Second floor square feet
# LowQualFinSF: Low quality finished square feet (all floors)
# GrLivArea: Above grade (ground) living area square feet
# GarageArea: Size of garage in square feet
# WoodDeckSF: Wood deck area in square feet
# OpenPorchSF: Open porch area in square feet
# EnclosedPorch: Enclosed porch area in square feet
# 3SsnPorch: Three season porch area in square feet
# ScreenPorch: Screen porch area in square feet
# PoolArea: Pool area in square feet
# MiscVal: $Value of miscellaneous feature

##### Numerical, Integer, Unbounded. ######
# YearBuilt: Original construction date
# YearRemodAdd: Remodel date (same as construction date if no remodeling or additions)
# BsmtFullBath: Basement full bathrooms
# BsmtHalfBath: Basement half bathrooms
# FullBath: Full bathrooms above grade
# HalfBath: Half baths above grade
# Bedroom: Bedrooms above grade (does NOT include basement bedrooms)
# Kitchen: Kitchens above grade
# TotRmsAbvGrd: Total rooms above grade (does not include bathrooms)
# Fireplaces: Number of fireplaces.
# GarageYrBlt: Year garage was built.
# GarageCars: Size of garage in car capacity
# MoSold: Month Sold (MM)
# YrSold: Year Sold (YYYY)

In [ ]:
#2. Features with missing values.
# LotFrontage, Alley, MasVnrType, MasVnrArea, BsmtQual, BsmtCond, BsmtExposure, BsmtFinType1, BsmtFinType2, Electrical, FireplaceQu, GarageType,
# GarageYrBlt, GarageFinish, GarageCars, GarageArea, GarageQual, GarageCond, PoolQC, Fence, MiscFeature 

# Some features are mostly missing values. These are:
# Alley, MasVnrType, FireplaceQu, PoolQC, Fence, MiscFeature
# To deal with them, the whole features could be dropped from the data set but in this case we will inpute the missing values depending in the 
# type of data. This is done for versatility/robustness.

# For rest of features, thier misssing values could be handled later by dropping the instances where they occur, but in this case they are also inputed.
# This is done because dropping the missing instances coudl bias the dataset in favor of more complete houses.

In [ ]:
#3. Some of the features on the data present outliers. These can be detected by visual in the histograms, where some of them are heavily skewed due to some
# extreme value. An example of this would be the features LotFrontage and LotArea. To identify outliers in a proper way some statistical tests would
# need to be conducted, like IQR and a boxplot.
_ = housing_df.hist(bins=50, figsize=(12, 8))
plt.tight_layout()

In [ ]:
#Divide the training and test sets.

#Stratified sampling
#Separating the sale prices in quantiles
quantiles = (
    housing_df.loc[:, "SalePrice"]
              .quantile(q=[0.1, 0.25, 0.5, 0.75, 0.9])
              .values
              .tolist()
)
#Creating a new feature categorizing the sale prices in their correspondig quantile
housing_df["sale_price_cat"] = pd.cut(
    housing_df["SalePrice"],
    bins=[0] + quantiles + [np.inf],
    labels=[1, 2, 3, 4, 5, 6]
)
#Spliting the data set with stratification based on the new feature.
seed = 42
random_state = np.random.RandomState(seed)

train_df, test_df = model_selection.train_test_split(
    housing_df,
    test_size=0.2,
    stratify=housing_df.loc[:, "sale_price_cat"],
    random_state=random_state
)
#Checking the distribution of both sets.
#Train set
_ = (
    train_df.loc[:, "sale_price_cat"]
              .value_counts()
              .sort_index()
              .plot
              .bar(rot=0, grid=True)
)
plt.xlabel("SalePrice category")
plt.ylabel("Number of houses")
plt.title("Train set")
plt.show()
#Test set
_ = (
    test_df.loc[:, "sale_price_cat"]
              .value_counts()
              .sort_index()
              .plot
              .bar(rot=0, grid=True)
)
plt.xlabel("SalePrice category")
plt.ylabel("Number of houses")
plt.title("Test set")
plt.show()

#Separate the train features and targets
train_features =train_df.drop(["SalePrice","sale_price_cat"], axis=1)
train_targets = train_df.loc[:, "SalePrice"]
#Separate the test features and the target
test_features = test_df.drop(["SalePrice","sale_price_cat"], axis=1)
test_targets = test_df.loc[:, "SalePrice"]

In [ ]:
#6. Visualize the data.
#Divide the training set in numerical and categorical for vizualization
train_features_num = train_df.select_dtypes(include=[np.int64,float])
train_features_cat = test_df.select_dtypes(include=object)

#Get column names
train_features_num_names = train_features_num.columns.tolist()
train_features_cat_names = train_features_cat.columns.tolist()

In [ ]:
#Plot numerical continous and categorical data
for name in train_features_num_names:
    _ = train_df.plot(
    kind="scatter",
    x=name,
    y="SalePrice",
    grid=True,
    alpha=0.25
    )

In [ ]:
#Plot categorical data
for name in train_features_cat_names:
    _ = (
    train_df.loc[:, [name, "SalePrice"]]
            .groupby(name)
            .mean()
            .plot(kind="bar", ylabel="House Price (USD)")
    )

In [ ]:
#4. What features appear useful for the task? Are there any that appear useless? Why?

#For the numerical data, most of it appears to be useful, as it shows some kind of trend when plotted against the sale price values. Among the most
# useful (presents clearest correlation) would be LotFrontage, LotArea and YearBuilt, 1stFlrSF, GrLivArea. Some not so useful continous numerical data
# would be OpenPorchSF and 3SsnPorch, as they don't show any meaningful correlation with sale prices.

#For categorical data some features like MoSold, YrSold and Utilities don't seem to contribute a lot of information for predicting sale value. While
# OverallQual and Full Bath present a very strong correlation with the target.

In [ ]:
#5. Identify and characterize the target distribution.
_ = (
    housing_df.loc[:, "SalePrice"]
              .hist()
)
plt.xlabel("SalePrice")
plt.ylabel("Number of houses")
plt.show()

print(housing_df.loc[:, "SalePrice"]
              .mean())
print(housing_df.loc[:, "SalePrice"]
              .std())

#The distribution of the target is positively skewed with a mean of 180921.1959 USD and standard deviation of 79442.5029 USD.

In [ ]:
#7. Study the correlations between attributes.
import seaborn as sns

corr_matrix = (
    train_df.drop("SalePrice", axis=1)
            .corr(numeric_only=True)
)
plt.figure(figsize=(12, 12))
_ = sns.heatmap(
    corr_matrix,
    annot=False,
    cmap="coolwarm",
    fmt=".2f",
    linewidths=0.5
)

plt.title("Feature Correlation Heatmap")
plt.show()

In [ ]:
#In the feature correlation heatmap we can see that most features are independet. Only a few of them present some slight correlation or anticorrelation
# between eachother. Like YearBuild and GarageYrBlt, for example, that are correlated to eachother.

# 4. Prepare the Data for ML (15 pts)

Use the Scikit Learn data pipeline and preprocessing API. Write functions for any custom data transformations used that are not available in the Scikit Learn API.

Minimum expectations. Some of these may or may not be necessary for this task. If a step is not necessary, discuss why.

1. Clean the data.
2. Perform feature selection.
3. Perform feature engineering.
5. Standardize or normalize features.

Your data preparation pipeline does not need to use all of the features, but you must use at least one categorical feature, one ordinal feature, and one numeric feature. These can be raw features, or features that you create yourself.

In [ ]:
#Function to transform int to float.
NUMERIC_FEATURES_DTYPES = {
    "LotArea": np.float64,
    "YearBuilt": np.float64,
    "YearRemodAdd": np.float64,
    "BsmtFinSF1": np.float64,
    "BsmtFinSF2": np.float64,
    "BsmtUnfSF": np.float64,
    "TotalBsmtSF": np.float64,
    "1stFlrSF": np.float64,
    "2ndFlrSF": np.float64,
    "LowQualFinSF": np.float64,
    "GrLivArea": np.float64,
    "GarageArea": np.float64,
    "WoodDeckSF": np.float64,
    "OpenPorchSF": np.float64,
    "EnclosedPorch": np.float64,
    "3SsnPorch": np.float64,
    "ScreenPorch": np.float64,
    "PoolArea": np.float64,
    "MiscVal": np.float64,
}


def convert_dtypes(df):
    return df.astype(
        NUMERIC_FEATURES_DTYPES
    )

In [ ]:
#Feature engineering.
def newFeatures(df):
    totFlrSF = (df["1stFlrSF"] + df["2ndFlrSF"]).clip(lower=-1e6, upper=1e6)
    totYear = (df["YearBuilt"] + df["YearRemodAdd"]).clip(lower=-1e6, upper=1e6)
    totPorchArea = (df["ScreenPorch"] + df["3SsnPorch"] + df["EnclosedPorch"] + df["OpenPorchSF"]).clip(lower=-1e6, upper=1e6)
    totLivnArea = (df["TotalBsmtSF"] + df["GrLivArea"] + df["GarageArea"]).clip(lower=-1e6, upper=1e6)
    bedrooms_to_totalrooms_ratio = (df["BedroomAbvGr"] / df["TotRmsAbvGrd"]).clip(lower=-1e6, upper=1)
    total_bathrooms = (df["FullBath"] + df["HalfBath"]).clip(lower=-1e6, upper=1e6)
    bathrooms_to_bedrooms_ratio = (total_bathrooms / df["BedroomAbvGr"]).clip(lower=-1e6, upper=1)

    new_feats = pd.DataFrame({
        "total_floor_SF": totFlrSF,
        "year_built_remodel": totYear,
        "total_porch_area": totPorchArea,
        "total_living_area": totLivnArea,
        "total_bathrooms": total_bathrooms,
        "bedrooms_to_totalrooms_ratio": bedrooms_to_totalrooms_ratio,
        "bathrooms_to_bedrooms_ratio": bathrooms_to_bedrooms_ratio,
        "bathrooms_to_totalrooms_ratio": bathrooms_to_bedrooms_ratio * bedrooms_to_totalrooms_ratio,
    }, index=df.index)  # keep the same row order

    return pd.concat([df, new_feats], axis=1)


In [ ]:
#Feature preparation pipeline
features_preparation_pipeline = pipeline.make_pipeline(
    preprocessing.FunctionTransformer(
        func=convert_dtypes,
    ),
    preprocessing.FunctionTransformer(
        func=newFeatures,
        validate=False
    ),
    compose.make_column_transformer(
        (preprocessing.OneHotEncoder(
                categories="auto",
                dtype=np.uint8,
                handle_unknown="infrequent_if_exist",
                max_categories=None,
                min_frequency=None,
                sparse_output=False,
            ),
            [
                "MSSubClass",
                "MSZoning",
                "Street",
                "LandContour",
                "LotConfig",
                "Neighborhood",
                "Condition1",
                "Condition2",
                "BldgType",
                "HouseStyle",
                "RoofStyle",
                "RoofMatl",
                "Exterior1st",
                "Exterior2nd",
                "Foundation",
                "Heating",
                "GarageType",
                "SaleType",
                "SaleCondition",
            ]
        ),
        (
        impute.SimpleImputer(
            strategy="constant",
            fill_value=0.0,
        ),
        [
            "LotFrontage",
            "MasVnrArea",
            "BsmtFinSF1",
            "BsmtFinSF2",
            "BsmtUnfSF",
            "TotalBsmtSF",
            "BsmtFullBath",
            "BsmtHalfBath",
            "Fireplaces",
            "GarageCars",
            "GarageArea",
            "WoodDeckSF",
            "OpenPorchSF",
            "EnclosedPorch",
            "3SsnPorch",
            "ScreenPorch",
            "PoolArea",
            "MiscVal",
        ]
        ),
        (
        impute.SimpleImputer(
            strategy="median",
        ),
            [
                "LotArea",
                "1stFlrSF",
                "2ndFlrSF",
                "LowQualFinSF",
                "GrLivArea",
                "total_floor_SF",
                "total_porch_area",
                "total_living_area",
                "bedrooms_to_totalrooms_ratio",
                "bathrooms_to_bedrooms_ratio",
                "bathrooms_to_totalrooms_ratio",
                "total_bathrooms",
            ]
        ),
        (
            preprocessing.OrdinalEncoder(
                categories=[
                    ["IR3", "IR2", "IR1", "Reg"],
                    #["ELO", "NoSeWa", "NoSewr", "AllPub"],
                    ["Sev", "Mod", "Gtl"],
                    ["Po", "Fa", "TA", "Gd", "Ex"],
                    ["Po", "Fa", "TA", "Gd", "Ex"],
                    ["Po", "Fa", "TA", "Gd", "Ex"],
                    ["Po", "Fa", "TA", "Gd", "Ex"],
                    ["No", "Mn", "Av", "Gd"],
                    ["Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],
                    ["Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],
                    ["Po", "Fa", "TA", "Gd", "Ex"],
                    ["N", "Y"],
                    ["Mix", "FuseP", "FuseF", "FuseA", "SBrkr"],
                    ["Po", "Fa", "TA", "Gd", "Ex"],
                    ["Sal", "Sev", "Maj2", "Maj1", "Min2", "Min1", "Typ"],
                    ["Po", "Fa", "TA", "Gd", "Ex"],
                    ["Unf", "RFn", "Fin"],
                    ["Po", "Fa", "TA", "Gd", "Ex"],
                    ["Po", "Fa", "TA", "Gd", "Ex"],
                    ["N", "P", "Y"],
                    ["Fa", "TA", "Gd", "Ex"],
                ],
                dtype=np.int8,
                encoded_missing_value=-1,
                handle_unknown="use_encoded_value",
                max_categories=None,
                min_frequency=None,
                unknown_value=-1,
            ),            
            [
                "LotShape",
                #"Utilities",
                "LandSlope",
                "ExterQual",
                "ExterCond",
                "BsmtQual",
                "BsmtCond",
                "BsmtExposure",
                "BsmtFinType1",
                "BsmtFinType2",
                "HeatingQC",
                "CentralAir",
                "Electrical",
                "KitchenQual",
                "Functional",
                "FireplaceQu",
                "GarageFinish",
                "GarageQual",
                "GarageCond",
                "PavedDrive",
                "PoolQC",
            ]
        ),
        (
        impute.SimpleImputer(
            strategy="most_frequent"
        ),
            [
                "OverallQual",
                "OverallCond",
                "YearBuilt",
                "YearRemodAdd",
                "FullBath",
                "HalfBath",
                "BedroomAbvGr",
                "KitchenAbvGr",
                "TotRmsAbvGrd",
                "MoSold",
                "YrSold",
                'GarageYrBlt',
                "year_built_remodel"
            ]
        ),
        remainder="drop",
        n_jobs=-1,
        verbose_feature_names_out=False,
    ),
    compose.make_column_transformer(
        (
            pipeline.make_pipeline(
                preprocessing.FunctionTransformer(
                    func=np.log1p,
                    inverse_func=np.expm1
                ),
                preprocessing.RobustScaler(
                    unit_variance=True
                ),
                verbose=True,
            ),
            [
                "LotArea",
                "MiscVal",
                "BsmtFinSF1",
                "BsmtFinSF2",
                "BsmtUnfSF",
                "TotalBsmtSF",
                "PoolArea",
                "LowQualFinSF",
                "WoodDeckSF",
                "OpenPorchSF",
                "EnclosedPorch",
                "3SsnPorch",
                "ScreenPorch",
            ]
        ),
        (
            pipeline.make_pipeline(
                preprocessing.RobustScaler(
                    unit_variance=True
                ),
                verbose=True,
            ),
            [
                "YearBuilt",
                "YearRemodAdd",
                "YrSold",
                'MasVnrArea',
                'LotFrontage',
                'GarageYrBlt',
                "1stFlrSF",
                "2ndFlrSF",
                "GrLivArea",
                "GarageArea",
                "total_floor_SF",
                "year_built_remodel",
                "total_porch_area",
                "total_living_area",
                "total_bathrooms",
                "bedrooms_to_totalrooms_ratio",
                "bathrooms_to_bedrooms_ratio",
                "bathrooms_to_totalrooms_ratio",
            ]
        ),
        n_jobs=1,
        remainder="passthrough",
        verbose=True,
        verbose_feature_names_out=True
    ),
    verbose=True,
).set_output(transform="pandas")

In [ ]:
features_preparation_pipeline

In [ ]:
prepared_train_features_df = (features_preparation_pipeline.fit_transform(train_features))
prepared_test_features_df = (features_preparation_pipeline.transform(test_features))

In [ ]:
prepared_train_features_df.info()
prepared_train_features_df.head()

In [ ]:
prepared_test_features_df.info()
prepared_test_features_df.head()

In [ ]:
#Quality Control
print(prepared_train_features_df.max().sort_values(ascending=False).head(10))

#Check correlation
corr_train_features_df = pd.concat([prepared_train_features_df,train_targets],axis = 1)
corr_test_features_df = pd.concat([prepared_test_features_df,test_targets],axis = 1)

print(corr_train_features_df.corr(numeric_only=True).loc[:, "SalePrice"]
      .sort_values(ascending=False))

In [ ]:
#This quality control shows that after transformation there are no extreme values in any of the columns. Also the correlation of the new features is good.

# 5. Explore Different Models (15 pts)

Minimum expectations:

1. Train an unregularized linear model, a linear model with default parameters, and a decision tree.
2. Measure and compare their performance.
3. Analyze the most significant variables for each algorithm.
4. Analyze the types of errors the models make.


In [ ]:
#Unregularized linear model
linear_model_pipeline = pipeline.make_pipeline(
    features_preparation_pipeline,
    compose.TransformedTargetRegressor(
        regressor=linear_model.SGDRegressor(penalty = None, max_iter = 5000, tol=1e-4,
                                            eta0 = 0.001),
        func=np.log1p,
        inverse_func = np.expm1
    )
)
#Model fitting and predictions
_ = linear_model_pipeline.fit(train_features, train_targets)
linear_train_predictions = linear_model_pipeline.predict(train_features)
linear_test_predictions = linear_model_pipeline.predict(test_features)

In [ ]:
#Train
linear_train_rmsle = metrics.mean_squared_log_error(train_targets,linear_train_predictions,)
linear_train_rmse = metrics.mean_squared_error(train_targets,linear_train_predictions,)
print(f"Unregularized Linear RMSLE on the training set: {linear_train_rmsle:.5f}")
print(f"Unregularized Linear RMSE on the training set: {linear_train_rmse:.5f}")
#Test
linear_test_rmsle = metrics.mean_squared_log_error(test_targets,linear_test_predictions,)
linear_test_rmse = metrics.mean_squared_error(test_targets,linear_test_predictions,)
print(f"Unregularized Linear RMSLE on the training set: {linear_test_rmsle:.5f}")
print(f"Unregularized Linear RMSE on the training set: {linear_test_rmse:.5f}")

In [ ]:
#Regularized linear model
regularized_linear_model_pipeline = pipeline.make_pipeline(
    features_preparation_pipeline,
    compose.TransformedTargetRegressor(
        regressor=linear_model.SGDRegressor(penalty='l2', alpha = 0.001,
                                            max_iter = 5000, tol=1e-4,eta0 = 0.001),
        func=np.log1p,
        inverse_func=np.expm1
    )
)
#Model fitting and predictions
_ = regularized_linear_model_pipeline.fit(train_features, train_targets)
regularized_train_predictions = regularized_linear_model_pipeline.predict(train_features)
regularized_test_predictions = regularized_linear_model_pipeline.predict(test_features)

In [ ]:
#Train
regularized_train_rmsle = metrics.mean_squared_log_error(train_targets,regularized_train_predictions,)
regularized_train_rmse = metrics.mean_squared_error(train_targets,regularized_train_predictions,)
print(f"Regularized Linear RMSLE on the training set: {regularized_train_rmsle:.5f}")
print(f"Regularized Linear RMSE on the training set: {regularized_train_rmse:.5f}")
#Test
regularized_test_rmsle = metrics.mean_squared_log_error(test_targets,regularized_test_predictions,)
regularized_test_rmse = metrics.mean_squared_error(test_targets,regularized_test_predictions,)
print(f"Regularized Linear RMSLE on the training set: {regularized_test_rmsle:.5f}")
print(f"Regularized Linear RMSE on the training set: {regularized_test_rmse:.5f}")

In [ ]:
_prng = np.random.RandomState(42)
#Decision tree
decision_tree_pipeline = pipeline.make_pipeline(
    features_preparation_pipeline,
    tree.DecisionTreeRegressor(
        random_state =_prng
    ),
)
#Model fitting and predictions
_ = decision_tree_pipeline.fit(train_features,train_targets)
tree_train_predictions = decision_tree_pipeline.predict(train_features)
tree_test_predictions = decision_tree_pipeline.predict(test_features)

In [ ]:
#Train
tree_train_rmsle = metrics.mean_squared_log_error(train_targets,tree_train_predictions,)
tree_train_rmse = metrics.mean_squared_error(train_targets,tree_train_predictions,)
print(f"Decision Tree RMSLE on the training set: {tree_train_rmsle:.5f}")
print(f"Decision Tree RMSE on the training set: {tree_train_rmse:.5f}")
#Test
tree_test_rmsle = metrics.mean_squared_log_error(test_targets,tree_test_predictions,)
tree_test_rmse = metrics.mean_squared_error(test_targets,tree_test_predictions,)
print(f"Decision Tree RMSLE on the training set: {tree_test_rmsle:.5f}")
print(f"Decision Tree RMSE on the training set: {tree_test_rmse:.5f}")

In [ ]:
feature_names = prepared_train_features_df.columns
#Unregularized Linear model
#Get feature coefficients.
reg = linear_model_pipeline.named_steps['transformedtargetregressor'].regressor_

#Put feature names and coefficients in a dataframe
coefs = pd.DataFrame({
    "feature": feature_names,
    "coefficient": reg.coef_
})
coefs["abs_coef"] = np.abs(coefs["coefficient"])
#Show top 20 coefficients
coefs = coefs.sort_values("abs_coef", ascending=False)
print(coefs.head(20))

In [ ]:
#Regularized Linear model
#Get feature coefficients.
reg = regularized_linear_model_pipeline.named_steps['transformedtargetregressor'].regressor_

#Put feature names and coefficients in a dataframe
coefs = pd.DataFrame({
    "feature": feature_names,
    "coefficient": reg.coef_
})
coefs["abs_coef"] = np.abs(coefs["coefficient"])
#Show top 20 coefficients
coefs = coefs.sort_values("abs_coef", ascending=False)
print(coefs.head(20))

In [ ]:
#Get tree model
tree_model = decision_tree_pipeline.named_steps["decisiontreeregressor"]
#Get important features
importances = pd.Series(tree_model.feature_importances_,
                        index=prepared_train_features_df.columns)
importances = importances.sort_values(ascending=False)
#20 Most important features
print(importances.head(20))

In [ ]:
#Plotting predictions
plt.subplot(2, 3, 1)
plt.scatter(np.log(linear_train_predictions),np.log(train_targets))
plt.subplot(2, 3, 2)
plt.scatter(np.log(tree_train_predictions),np.log(train_targets))
plt.subplot(2, 3, 3)
plt.scatter(np.log(regularized_train_predictions),np.log(train_targets))
plt.subplot(2, 3, 4)
plt.scatter(linear_train_predictions,train_targets)
plt.subplot(2, 3, 5)
plt.scatter(tree_train_predictions,train_targets)
plt.subplot(2, 3, 6)
plt.scatter(regularized_train_predictions,train_targets)

# 6. Fine-tune the Models (15 pts)

1. Fine-tune the hyperparameters for a linear model using RandomizedSearchCV.
2. Fine-tune the hyperparameters for a decision tree using GridSearchCV.


In [ ]:
root_mean_square_log_error_scoring = metrics.make_scorer(
    metrics.root_mean_squared_log_error,
    greater_is_better=False
)

In [ ]:
from sklearn.linear_model import ElasticNetCV
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import KFold
from sklearn.model_selection import GridSearchCV

cv_splits = KFold(n_splits=5, shuffle=True, random_state=_prng)

#Regularized linear model
linear_param_grid = {
    "transformedtargetregressor__regressor__alpha": [0.00001, 0.0001, 0.001, 0.01],
    "transformedtargetregressor__regressor__penalty": ["l2", "l1", "elasticnet"],
    "transformedtargetregressor__regressor__eta0": [0.0001, 0.001, 0.01],
    "transformedtargetregressor__regressor__l1_ratio": [0.15, 0.5, 0.85]  # only used if penalty='elasticnet'
}

linear_grid_search = GridSearchCV(
    estimator=linear_model_pipeline,
    param_grid=linear_param_grid,
    scoring=root_mean_square_log_error_scoring,
    cv=5,
    n_jobs=-1,
    verbose=0
)

linear_grid_search.fit(train_features, train_targets)

In [ ]:
print("Best parameters:", linear_grid_search.best_params_)
print("Best CV score:", -linear_grid_search.best_score_)

In [ ]:
#Decision Tree
tree_param_grid = {
    'decisiontreeregressor__max_depth': [5, 10, 15, None],
    'decisiontreeregressor__min_samples_split': [2, 5, 10],
    'decisiontreeregressor__min_samples_leaf': [1, 2, 4]
}

tree_grid_search = GridSearchCV(
    decision_tree_pipeline,
    tree_param_grid,
    cv=5,
    scoring=root_mean_square_log_error_scoring,
    n_jobs=-1,
    verbose=0
)

tree_grid_search.fit(train_features, train_targets)

In [ ]:
print("Best params:", tree_grid_search.best_params_)
print("Best RMSLE:", -tree_grid_search.best_score_)

In [ ]:
#Best Regularized linear model
regularized_linear_model_pipeline = pipeline.make_pipeline(
    features_preparation_pipeline,
    compose.TransformedTargetRegressor(
        regressor=linear_model.SGDRegressor(penalty='l2', alpha = 0.001,
                                            max_iter = 5000, tol=1e-4,eta0 = 0.01),
        func=np.log1p,
        inverse_func=np.expm1
    )
)
#Model fitting and predictions
_ = regularized_linear_model_pipeline.fit(train_features, train_targets)
regularized_train_predictions = regularized_linear_model_pipeline.predict(train_features)
regularized_test_predictions = regularized_linear_model_pipeline.predict(test_features)

In [ ]:
#Train
regularized_train_rmsle = metrics.mean_squared_log_error(train_targets,regularized_train_predictions,)
regularized_train_rmse = metrics.mean_squared_error(train_targets,regularized_train_predictions,)
print(f"Regularized Linear RMSLE on the training set: {regularized_train_rmsle:.5f}")
print(f"Regularized Linear RMSE on the training set: {regularized_train_rmse:.5f}")
#Test
regularized_test_rmsle = metrics.mean_squared_log_error(test_targets,regularized_test_predictions,)
regularized_test_rmse = metrics.mean_squared_error(test_targets,regularized_test_predictions,)
print(f"Regularized Linear RMSLE on the training set: {regularized_test_rmsle:.5f}")
print(f"Regularized Linear RMSE on the training set: {regularized_test_rmse:.5f}")

In [ ]:
#Best decision tree model
decision_tree_pipeline = pipeline.make_pipeline(
    features_preparation_pipeline,
    tree.DecisionTreeRegressor(
        max_depth=5,
        min_samples_leaf=1,  # remove the underscore
        min_samples_split=10,
        random_state=_prng
    ),
)
#Model fitting and predictions
_ = decision_tree_pipeline.fit(train_features,train_targets)
tree_train_predictions = decision_tree_pipeline.predict(train_features)
tree_test_predictions = decision_tree_pipeline.predict(test_features)

In [ ]:
#Train
tree_train_rmsle = metrics.mean_squared_log_error(train_targets,tree_train_predictions,)
tree_train_rmse = metrics.mean_squared_error(train_targets,tree_train_predictions,)
print(f"Decision Tree RMSLE on the training set: {tree_train_rmsle:.5f}")
print(f"Decision Tree RMSE on the training set: {tree_train_rmse:.5f}")
#Test
tree_test_rmsle = metrics.mean_squared_log_error(test_targets,tree_test_predictions,)
tree_test_rmse = metrics.mean_squared_error(test_targets,tree_test_predictions,)
print(f"Decision Tree RMSLE on the training set: {tree_test_rmsle:.5f}")
print(f"Decision Tree RMSE on the training set: {tree_test_rmse:.5f}")

In [ ]:
#Results
#The cross validation found better hyperparameter combinations for both the regularized linear model and the decision tree. The regularized linear model
#now performs 

In [ ]:
#Plotting predictions
plt.subplot(2, 3, 1)
plt.scatter(np.log(linear_train_predictions),np.log(train_targets))
plt.subplot(2, 3, 2)
plt.scatter(np.log(tree_train_predictions),np.log(train_targets))
plt.subplot(2, 3, 3)
plt.scatter(np.log(regularized_train_predictions),np.log(train_targets))
plt.subplot(2, 3, 4)
plt.scatter(linear_train_predictions,train_targets)
plt.subplot(2, 3, 5)
plt.scatter(tree_train_predictions,train_targets)
plt.subplot(2, 3, 6)
plt.scatter(regularized_train_predictions,train_targets)

# 7. Summarize your results (15 pts)

1. Provide a 2-3 sentence summary of your solution for a non-technical audience of real estate executives.

2. Provide a technical summary of your solution for the ML team lead at a real estate investment company.

In [ ]:
#1. To predict house sale prices some machine learning models were trained based on the available information on the houses sold in the past. These we
# a mathematical model and a decision based model. They showed to be capable of predicting the sale prices successfully.

#2. Based on the given numerical and categorical features, some new features were engineered that showed promising correlation. Then the whole data set
# was transformed to numerical values, imputed and rescaled to be the input of the machine learning models.
# Three models were fitted, an inregularized linear model, a regularized linear model and a decision tree. All of these were trained using a stochastic
# gradient descent regressor. All of them converged to a solution. The liear models performed decently and the decision tree managed to memorize the
# training dataset, while performing well on the test set.
# Then, the hyperparameters of the regularized linear model and the decision tree were optimized through cross validation. Better versions of these model
# were found, fitted and measured. The updated regularized linear model performed significantly better and the updated decision tree stopped overfiting
# to the training data while keeping the best performance of the three models on the test data.

#The final model chosen for the solution upload is the decision tree.

In [ ]:
#1. Loading the data.
housing_test_df = pd.read_csv(
    "/kaggle/input/house-prices-advanced-regression-techniques/test.csv",
    index_col="Id",
)

In [ ]:
#Quality check
final_test_features_df = (features_preparation_pipeline.transform(housing_test_df))
final_test_features_df.info()
final_test_features_df.head()

In [ ]:
#Model fitting and predictions
final_predictions = decision_tree_pipeline.predict(housing_test_df)

In [ ]:
test_predictions = final_predictions

In [ ]:
submission_df = (
    pd.read_csv(
        "/kaggle/input/house-prices-advanced-regression-techniques/sample_submission.csv",
        index_col="Id"
    ).assign(
        SalePrice=test_predictions
    ).to_csv(
        "submission.csv",
        index=True,
    )
)

In [ ]:
%%bash

cat submission.csv | head -n 5